# DTP Vaccination rates and attrition using DHS data

Ce rapport génère des visualisations pour les indicateurs relatifs à la vaccination des enfants avec le vaccin contre la diphtérie (D), tétanos (T) et coqueluche (P), à partir des données de l'Enquête Démographique et de Santé (EDS - DHS).

---

* *Numérateur* : Nombre d'enfants qui sont membres de fait du ménage, testés par test de diagnostic rapide (TDR) ou par microscopie, et dont le résultat est positif pour le paludisme.
* *Dénominateur* : Nombre d'enfants qui sont membres de fait du ménage, en âge d'avoir reçu la dose de vaccin en question.

---

Pour plus d'informations (en anglais):
- Ressources relatives à la vaccination DTP chez les enfants
    - [Définition et calculs](https://dhsprogram.com/data/Guide-to-DHS-Statistics/index.htm#t=Vaccination.htm%23Percentage_of_childrenbc-1&rhtocid=_13_1_0)
- [Les questionnaires utilisés dans les EDS/DHS](https://dhsprogram.com/publications/publication-dhsg4-dhs-questionnaires-and-manuals.cfm)

---

*Note* : Contrairement à la majorité des analyses dans le cadre du processus SNT, cette analyse est menée au niveau administratif **ADM1**, en raison de la disponibilité des données.

## Preliminaries

In [ ]:
rm(list = ls())

options(scipen=999)

In [ ]:
# Global paths
Sys.setenv(PROJ_LIB = "/opt/conda/share/proj")
Sys.setenv(GDAL_DATA = "/opt/conda/share/gdal")

In [ ]:
# Paths
ROOT_PATH <- '~/workspace'
PIPELINE_PATH <- file.path(ROOT_PATH, 'pipelines', 'snt_dhs_indicators')

In [ ]:
# Load notebook-specific utils
source(file.path(PIPELINE_PATH, "utils", "snt_dhs_vaccination_computation.r"))

setup_ctx <- bootstrap_dhs_indicators_context(
  root_path = ROOT_PATH,
  required_packages = c("readr", "haven", "glue", "survey", "data.table", "sf", "ggplot2", "stringi", "reticulate", "jsonlite", "httr", "arrow")
)

DATA_PATH <- setup_ctx$DATA_PATH
DHS_DATA_PATH <- setup_ctx$DHS_DATA_PATH
config_json <- setup_ctx$config_json
COUNTRY_CODE <- setup_ctx$COUNTRY_CODE
OUTPUT_DATA_PATH <- file.path(DATA_PATH, 'dhs', 'indicators', 'vaccination')
dir.create(OUTPUT_DATA_PATH, recursive = TRUE, showWarnings = FALSE)

In [ ]:
reticulate::py_config()$python

In [ ]:
# Configuration already loaded by bootstrap_dhs_indicators_context().

In [ ]:
# Set config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE

## Geo data

In [ ]:
admin_level <- 'ADM1'
admin_id_col <- glue(admin_level, 'ID', .sep='_')
admin_name_col <- glue(admin_level, 'NAME', .sep='_')
admin_cols <- c(admin_id_col, admin_name_col)

In [ ]:
# Load spatial file from dataset

dhis2_dataset <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED

spatial_data <- load_dhs_spatial_data(
  dhis2_dataset = dhis2_dataset,
  country_code = COUNTRY_CODE
)

spatial_data <- st_as_sf(spatial_data)

# aggregate geometries by the admin columns
spatial_data <- aggregate_geometry(
  sf_data=spatial_data,
  admin_id_colname=admin_id_col,
  admin_name_colname=admin_name_col
  )

# keep class
spatial_data <- st_as_sf(spatial_data)

# DRC provinces need to be cleaned
if(COUNTRY_CODE == "COD"){
  spatial_data[[admin_name_col]] <- clean_admin_names(spatial_data[[admin_name_col]])
}

admin_data <- st_drop_geometry(spatial_data)
setDT(admin_data)

## Import DHS data

In [ ]:
vaccination_doses <- c(1, 2, 3)
indicator_access <- 'PCT_DTP'
indicator_attrition <- 'PCT_DROPOUT_DTP'

In [ ]:
data_source <- 'DHS'
household_recode <- 'HR'
kid_recode <- 'KR'
target_file_type <- 'SV'

delete_otherextension_files(DHS_DATA_PATH, extension_to_retain=".zip")

In [ ]:
dhs_hr_zip_filename <- extract_latest_dhs_recode_filename(DHS_DATA_PATH, household_recode, target_file_type)
unzip(file.path(DHS_DATA_PATH, dhs_hr_zip_filename), exdir=DHS_DATA_PATH)

dhs_kr_zip_filename <- extract_latest_dhs_recode_filename(DHS_DATA_PATH, kid_recode, target_file_type)
unzip(file.path(DHS_DATA_PATH, dhs_kr_zip_filename), exdir=DHS_DATA_PATH)

# # Remove existing output files
# files <- list.files(OUTPUT_DATA_PATH, full.names = TRUE)
# files_to_delete <- files[grepl('DTP', basename(files), ignore.case = TRUE) & grepl(COUNTRY_CODE, basename(files), ignore.case = TRUE)]
# file.remove(files_to_delete)

data_extension <- '.SAV'
dhs_hr_filename <- list.files(path = DHS_DATA_PATH, pattern = paste0(".*", household_recode, ".*\\", data_extension, "$"), ignore.case=TRUE)
dhs_kr_filename <- dir(path = DHS_DATA_PATH, pattern = paste0(".*", kid_recode, ".*\\", data_extension, "$"), ignore.case=TRUE)

if(!check_dhs_same_version(dhs_hr_filename, dhs_kr_filename)){
  stop("The necessary DHS data do not have the same version/issue. Check available data before rerunning.")
}

dhs_hr_dt <- read_spss(file.path(DHS_DATA_PATH, dhs_hr_filename)) # household recode
dhs_hr_dt <- setDT(dhs_hr_dt)

dhs_kr_dt <- read_spss(file.path(DHS_DATA_PATH, dhs_kr_filename)) # kid recode
dhs_kr_dt <- setDT(dhs_kr_dt)

## Preprocess DHS data

### Extract DHS admin data

In [ ]:
# Make admin codes and names dataframe (for future merging)

dhs_beginning_year <- as.integer(dhs_hr_dt[, min(HV007)])

dhs_admin_dt <- make_dhs_admin_df(
  input_dhs_df=dhs_hr_dt,
  original_admin_column="HV024",
  new_admin_name_colname=admin_name_col,
  new_admin_code_colname='DHS_ADM1_CODE'
)

# format the names to be like DHIS2 names
dhs_admin_dt[, (admin_name_col) := format_names(get(admin_name_col))]

# TODO this should be changed in the formatting of DHIS2 data; the correct name should be with a space
dhs_admin_dt[get(admin_name_col) == "MAI NDOMBE", (admin_name_col) := "MAINDOMBE"]

# Check that all regions can be matched with DHIS2 pyramid
if(!check_perfect_match(dhs_admin_dt, admin_name_col, admin_data, admin_name_col)){
  stop("The DHS data provided does not fully match DHIS2 pyramid data. Please check input data before retrying.")
}

rm(dhs_hr_dt) # free up resources

### Filter rows and columns

In [ ]:
# remove dead children from the dataset, keep only children aged 1 or more (avoid left censoring for vaccination) and respect the base for the 'h' variables
kr_dt <- dhs_kr_dt[B5 == 1 & B8 >= 1 & B19 < 36,]

household_id_cols <- c('V000', 'V001', 'V002')
kid_id_cols <- c('CASEID', 'BIDX')
kid_dpt1_cols <- c('H3', 'H3D', 'H3M', 'H3Y')
kid_dpt2_cols <- c('H5', 'H5D', 'H5M', 'H5Y')
kid_dpt3_cols <- c('H7', 'H7D', 'H7M', 'H7Y')
kid_sampling_cols <- c('V005', 'V021', 'V023', 'V024')

kr_dt <- kr_dt[, .SD, .SDcols = c(household_id_cols, kid_id_cols, kid_sampling_cols, kid_dpt1_cols, kid_dpt2_cols, kid_dpt3_cols)]

# # check i didn't omit any crucial variable
# stopifnot(nrow(kr_dt[duplicated(kr_dt)]) == 0)

### New features

Add the region labels, to subsequently match DHIS2 data

In [ ]:
kr_dt <- merge.data.table(dhs_admin_dt, kr_dt, by.x = "DHS_ADM1_CODE", by.y = "V024", all = TRUE)

Create the target features (whether or not the kid was vaccinated, for each dose)

In [ ]:
# Create dummy variables for the various DTP vaccine doses
kr_dt[, `:=`(
  DTP1 = fcase(
    H3 == 0L, 0L,
    H3 %in% c(1L, 2L, 3L), 1L,
    default = NA
  ),
  DTP2 = fcase(
    H5 == 0L, 0L,
    H5 %in% c(1L, 2L, 3L), 1L,
    default = NA
  ),
  DTP3 = fcase(
    H7 == 0L, 0L,
    H7 %in% c(1L, 2L, 3L), 1L,
    default = NA
  )
)]

# Correct external consistency issues: children who got the third dose also had the second, and so on:
kr_dt[DTP2 == 1, DTP1 := 1]
kr_dt[DTP3 == 1, DTP1 := 1]
kr_dt[DTP3 == 1, DTP2 := 1]

### Create the survey design

In [ ]:
# compute the household/kid weights
kr_dt[, wt := V005/1000000]

In [ ]:
# account for the sampling strategy (clustering, stratification, weights) for means, proportions, regression models, etc.
dtp_design = svydesign(
  ids = ~ V021, # primary sampling unit / cluster ids (cluster number and/or ultimate area unit)
  data = kr_dt, # dataset
  strata = ~ V023, # groupings of primary sampling units
  weights = ~ wt, # the sampling weights variable
  nest = T # the primary sampling units are nested within the strata
  )

## Vaccination proportion indicator

For each vaccine dose:
- compute the proportions of vaccinated per region
- compute the CIs
- add the admin units and save to .csv and parquet

In [ ]:
vaccination_results <- compute_dtp_indicator_tables(
  dtp_design = dtp_design,
  vaccination_doses = vaccination_doses,
  indicator_access = indicator_access,
  admin_name_col = admin_name_col,
  admin_cols = admin_cols,
  admin_data = admin_data,
  output_data_path = OUTPUT_DATA_PATH,
  country_code = COUNTRY_CODE,
  data_source = data_source,
  admin_level = admin_level
)

DTP_DROPOUT <- vaccination_results$dtp_dropout
PCT_DTP1 <- vaccination_results$dose_tables[["PCT_DTP1"]]
PCT_DTP2 <- vaccination_results$dose_tables[["PCT_DTP2"]]
PCT_DTP3 <- vaccination_results$dose_tables[["PCT_DTP3"]]

## Dropout rate indicator

Add dropout rates plots: for each vaccine dose:
- make the dropout rates
- add them to the summary file and save it as .csv and parquet
- make plots and save them

In [ ]:
# dropout computed and exported in next cell using helper

In [ ]:
DTP_DROPOUT <- compute_and_export_dtp_dropout(
  dtp_dropout = DTP_DROPOUT,
  vaccination_doses = vaccination_doses,
  indicator_access = indicator_access,
  indicator_attrition = indicator_attrition,
  output_data_path = OUTPUT_DATA_PATH,
  country_code = COUNTRY_CODE,
  data_source = data_source,
  admin_level = admin_level
)